# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
# your code here

# Step 1: Data Engineering

# 1. Read tic-tac-toe.csv into a dataframe.
import pandas as pd

df = pd.read_csv('tic-tac-toe.csv')

# 2. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
print(df.head())  # View first few rows
print(df.info())  # Check data types and missing values
print(df.describe(include='all'))  # Summary statistics
print(df['class'].value_counts())  # Check class distribution

# Reliability check: 
# - No missing values (from info).
# - Categorical values are consistent: 'x', 'o', 'b' in grid columns, True/False in class.
# - Dataset seems reliable as it matches the description: 9 grid positions + class label.
# - Total rows: 958 (standard Tic Tac Toe endgame dataset size), no apparent duplicates or anomalies.

# 3. Convert the categorical values to numeric in all columns.
# Map 'x' -> 1, 'o' -> -1, 'b' -> 0 (common encoding for Tic Tac Toe).
# Map class: True -> 1, False -> 0.
mapping = {'x': 1, 'o': -1, 'b': 0}
for col in df.columns[:-1]:  # Apply to all grid columns
    df[col] = df[col].map(mapping)

df['class'] = df['class'].astype(int)  # True/False to 1/0

# Verify conversion
print(df.head())

# 4. Separate the inputs and output.
X = df.drop('class', axis=1)  # Inputs: grid positions
y = df['class']  # Output: class

# 5. Normalize the input data.
# Use MinMaxScaler to scale between 0 and 1 (since values are -1, 0, 1).
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Convert back to DataFrame for convenience (optional)
X_normalized = pd.DataFrame(X_normalized, columns=X.columns)

# Verify normalization
print(X_normalized.head())

  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
None
         TL   TM   TR   ML   MM   MR   BL   BM   BR class
count   958  958  958  958  958  958  958  958  958   958
unique    3    3    3    3    3    3    3    3    3     2
top       x    x  

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [5]:
# your code here

# Importaciones actualizadas y recomendadas (2024-2025)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 2. Modelo
model = Sequential([
    Dense(64, activation='relu', input_shape=(9,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2, activation='softmax')
])

model.summary()

# 3. Compilar
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Entrenar
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# 5. Evaluar
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# 6. Guardar
model.save('tic-tac-toe.model.keras')
print("Modelo guardado como 'tic-tac-toe.model'")

TensorFlow version: 2.20.0


c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,282 (12.82 KB)

 Trainable params: 3,282 (12.82 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5196 - loss: 0.6954 - val_accuracy: 0.6364 - val_loss: 0.6488
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6585 - loss: 0.6387 - val_accuracy: 0.6364 - val_loss: 0.6245
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6585 - loss: 0.6079 - val_accuracy: 0.6364 - val_loss: 0.6050
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6601 - loss: 0.5837 - val_accuracy: 0.6364 - val_loss: 0.5841
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6667 - loss: 0.5609 - val_accuracy: 0.7013 - val_loss: 0.5636
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7304 - loss: 0.5374 - val_accuracy: 0.7273 - val_loss: 0.5513
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7647 - loss: 0.5208 - val_accuracy: 0.7468 - val_loss: 0.5325
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7892 - loss: 0.4963 - val_accuracy: 0.7532 - val_loss

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [11]:
from tensorflow.keras.models import load_model
import numpy as np

# Cargar el modelo con el nombre REAL que tienes
model = load_model('tic-tac-toe.model.keras')

print("¡Modelo cargado correctamente! ✓\n")

# Resto del código de predicciones (ejemplo con 5 filas aleatorias)
idx = np.random.choice(len(X_test), size=5, replace=False)
samples = X_test.iloc[idx] if hasattr(X_test, 'iloc') else X_test[idx]
true_labels = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]

probs = model.predict(samples, verbose=0)
predicted = np.argmax(probs, axis=1)

print("Resultados de 5 predicciones aleatorias:")
for i, (pred, real) in enumerate(zip(predicted, true_labels), 1):
    print(f"Ejemplo {i:2d}:  Predicción = {pred}   |   Real = {real}   → {'✓ Correcto' if pred == real else '✗ Error'}")

¡Modelo cargado correctamente! ✓

Resultados de 5 predicciones aleatorias:
Ejemplo  1:  Predicción = 1   |   Real = 1   → ✓ Correcto
Ejemplo  2:  Predicción = 0   |   Real = 0   → ✓ Correcto
Ejemplo  3:  Predicción = 1   |   Real = 1   → ✓ Correcto
Ejemplo  4:  Predicción = 1   |   Real = 1   → ✓ Correcto
Ejemplo  5:  Predicción = 1   |   Real = 1   → ✓ Correcto


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [12]:
# your code here

# Step 4: Improve Your Model
# -------------------------------------------------------

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np

# Suponemos que ya tienes X_normalized e y preparados del Step 1
# Si no, recuerda cargarlos y preprocesarlos

# Volvemos a dividir los datos (por si acaso)
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# -------------------------------
# Modelo BASE (el que ya tenías - referencia)
# -------------------------------
print("=== MODELO BASE (referencia) ===")
model_base = Sequential([
    Dense(64, activation='relu', input_shape=(9,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2, activation='softmax')
])

model_base.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_base = model_base.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_base, test_acc_base = model_base.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy base: {test_acc_base:.4f}  |  Test loss: {test_loss_base:.4f}\n")


# -------------------------------
# Experimento 1: MÁS CAPAS
# -------------------------------
print("=== Experimento 1: Más capas (más profundo) ===")
model_deep = Sequential([
    Dense(128, activation='relu', input_shape=(9,)),
    Dense(64,  activation='relu'),
    Dense(32,  activation='relu'),
    Dense(16,  activation='relu'),
    Dense(8,   activation='relu'),     # capa extra
    Dense(2,   activation='softmax')
])

model_deep.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_deep.fit(
    X_train, y_train,
    epochs=60,                # un poco más de épocas para darle chance
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_deep, test_acc_deep = model_deep.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy (más capas): {test_acc_deep:.4f}  |  Test loss: {test_loss_deep:.4f}")
print(f"Δ respecto al base: {test_acc_deep - test_acc_base:+.4f}\n")


# -------------------------------
# Experimento 2: Learning rate personalizado
# -------------------------------
print("=== Experimento 2: Learning rate más bajo ===")
model_lr = Sequential([
    Dense(64, activation='relu', input_shape=(9,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(2, activation='softmax')
])

# Learning rate más pequeño → aprendizaje más fino y estable
optimizer_custom = Adam(learning_rate=0.0005)   # default suele ser ~0.001

model_lr.compile(
    optimizer=optimizer_custom,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_lr.fit(
    X_train, y_train,
    epochs=80,                    # más épocas porque va más lento
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_lr, test_acc_lr = model_lr.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy (lr=0.0005): {test_acc_lr:.4f}  |  Test loss: {test_loss_lr:.4f}")
print(f"Δ respecto al base: {test_acc_lr - test_acc_base:+.4f}\n")


# -------------------------------
# Experimento 3: Combinación (más capas + lr ajustado + más épocas)
# -------------------------------
print("=== Experimento 3: Combinación (mejor de ambos mundos?) ===")
model_best = Sequential([
    Dense(128, activation='relu', input_shape=(9,)),
    Dense(64,  activation='relu'),
    Dense(32,  activation='relu'),
    Dense(16,  activation='relu'),
    Dense(2,   activation='softmax')
])

optimizer_best = Adam(learning_rate=0.0003)

model_best.compile(
    optimizer=optimizer_best,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_best.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_best, test_acc_best = model_best.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy (combinación): {test_acc_best:.4f}  |  Test loss: {test_loss_best:.4f}")
print(f"Δ respecto al base: {test_acc_best - test_acc_base:+.4f}\n")


# -------------------------------
# Resumen final
# -------------------------------
print("┌─────────────────────────────┬───────────────┬──────────────┐")
print("│ Modelo                      │ Test Accuracy │ Test Loss    │")
print("├─────────────────────────────┼───────────────┼──────────────┤")
print(f"│ Base                        │ {test_acc_base:.4f}        │ {test_loss_base:.4f} │")
print(f"│ + Más capas                 │ {test_acc_deep:.4f}        │ {test_loss_deep:.4f} │")
print(f"│ + Learning rate bajo        │ {test_acc_lr:.4f}        │ {test_loss_lr:.4f} │")
print(f"│ Combinación (mejor intento) │ {test_acc_best:.4f}        │ {test_loss_best:.4f} │")
print("└─────────────────────────────┴───────────────┴──────────────┘")

print("\n¿Cuál fue la mejor configuración en tu caso?")

=== MODELO BASE (referencia) ===


c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Test accuracy base: 0.9740  |  Test loss: 0.1336

=== Experimento 1: Más capas (más profundo) ===


c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Test accuracy (más capas): 0.9792  |  Test loss: 0.1225
Δ respecto al base: +0.0052

=== Experimento 2: Learning rate más bajo ===


c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Test accuracy (lr=0.0005): 0.9792  |  Test loss: 0.1023
Δ respecto al base: +0.0052

=== Experimento 3: Combinación (mejor de ambos mundos?) ===


c:\Users\angel\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Test accuracy (combinación): 0.9844  |  Test loss: 0.0997
Δ respecto al base: +0.0104

┌─────────────────────────────┬───────────────┬──────────────┐
│ Modelo                      │ Test Accuracy │ Test Loss    │
├─────────────────────────────┼───────────────┼──────────────┤
│ Base                        │ 0.9740        │ 0.1336 │
│ + Más capas                 │ 0.9792        │ 0.1225 │
│ + Learning rate bajo        │ 0.9792        │ 0.1023 │
│ Combinación (mejor intento) │ 0.9844        │ 0.0997 │
└─────────────────────────────┴───────────────┴──────────────┘

¿Cuál fue la mejor configuración en tu caso?


**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
# your answer here